Projekat: klasifikacija vesti (fake / real).

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [6]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip


Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100%|█████████████████████████████████████| 41.0M/41.0M [00:03<00:00, 13.3MB/s]



In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Download necessary NLTK data
nltk.download('stopwords')

# LOAD DATA

fake = pd.read_csv('./data/Fake.csv')
true = pd.read_csv('./data/True.csv')

fake['label'] = 1
true['label'] = 0

df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

df = df[['text', 'label']]

# PREPROCESSING FUNCTION
ps = PorterStemmer()

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    words = [ps.stem(word) for word in words if word not in stop_words]
    
    return ' '.join(words)

print("Cleaning text... (this may take a few minutes)")
df['text'] = df['text'].apply(clean_text)
print("Cleaning complete!")

# SPLIT DATA
X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# VECTORIZATION (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# MODEL TRAINING & EVALUATION

models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": LinearSVC()
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, predictions)
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, predictions))
    results[name] = acc

# FINAL COMPARISON
print("\n" + "="*30)
print("FINAL MODEL COMPARISON")
print("="*30)
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

[nltk_data] Downloading package stopwords to /home/uno/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaning text... (this may take a few minutes)
Cleaning complete!

--- Training Logistic Regression ---
Accuracy: 0.9846
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      4270
           1       0.99      0.98      0.99      4710

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980


--- Training KNN ---
Accuracy: 0.7399
              precision    recall  f1-score   support

           0       0.91      0.50      0.65      4270
           1       0.68      0.96      0.79      4710

    accuracy                           0.74      8980
   macro avg       0.80      0.73      0.72      8980
weighted avg       0.79      0.74      0.72      8980


--- Training Linear SVM ---
Accuracy: 0.9931
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4270
           1       0.99      0.